# Stage1 09:46 同业早盘广度增量信息
冻结原 39 项与一个新坐标；本单元仅做来源、成熟度、训练新颖性及 DatasetH 无拟合预检。

In [1]:
import hashlib,json,os
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd().resolve();card=root/'research/stage1_0946_peer_breadth_relative_h5_information_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
parent=plan['parent'];pc=root/parent['source_card']
for artifact,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:
 assert sha(pc/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/artifact)==parent[key]
assert json.loads((root/parent['nas_readback']).read_text())['NAS_archive_sha256']==parent['nas_archive_sha256']
src=plan['source'];sc=root/src['card'];ss=json.loads((sc/'summary.json').read_text())
assert sha(sc/'summary.json')==src['summary_sha256'] and ss['recorder_id']==src['recorder_id'] and ss['experiment_id']==src['experiment_id'] and ss['nas_readback']['verified']
assert sha(sc/'qlib_recorder'/src['experiment_id']/src['recorder_id']/'artifacts'/src['feature_artifact'])==src['feature_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(pc/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns) and frame.index.equals(base.index) and frame.index.equals(timing.index)
with R.uri_context(str(sc/'qlib_recorder')):
 mapped=R.get_recorder(recorder_id=src['recorder_id'],experiment_id=src['experiment_id']).load_object(src['feature_artifact'])
assert len(mapped)==2006 and mapped.trade_date.is_unique and mapped.source_known.all()
assert (pd.to_datetime(mapped.assumed_available_at)<pd.to_datetime(mapped.receiver_decision_at)).all()
feature=mapped.set_index('trade_date')['peer_positive_fraction_3'].astype(float).reindex(frame.index).rename(plan['single_mechanism']['feature'])
def h5(series):
 parts=[series.shift(-i) for i in range(5)]
 valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
 return pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=series.index)
label=(h5(frame.soxl_forward_open_return)-h5(frame.soxx_forward_open_return)).rename('SOXL_relative_H5')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
train_cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');held_cutoff=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
cohort=plan['cohorts'];train_dates=(frame.index>=pd.Timestamp(cohort['train_signal'][0]))&(frame.index<=pd.Timestamp(cohort['train_signal'][1]))&(mature<train_cutoff)&label.notna();held_dates=(frame.index>=pd.Timestamp(cohort['held_reused_diagnostic'][0]))&(frame.index<=pd.Timestamp(cohort['held_reused_diagnostic'][1]))&(mature<held_cutoff)&label.notna()
assert int(train_dates.sum())>=cohort['minimum_mature_train_rows'] and int(held_dates.sum())>=cohort['minimum_mature_held_rows']
assert feature.loc[train_dates|held_dates].notna().all() and np.isfinite(feature.loc[train_dates|held_dates]).all()
maxcorr=float(base.loc[train_dates,base_names].corrwith(feature.loc[train_dates]).abs().max())
assert maxcorr<plan['single_mechanism']['training_only_max_abs_pairwise_Pearson_stop_at_or_above']
aug=base[base_names].copy();aug[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates},index=frame.index)
static=pd.concat({'feature':aug,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(cohort['train_signal']),'held':tuple(cohort['held_reused_diagnostic'])})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(tr)>=int(train_dates.sum()) and len(he)>=int(held_dates.sum())
report={'status':'conditional_0946_peer_H5_DatasetH_source_maturity_novelty_preflight_passed_no_fit','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['recorder_id'],'parent_recorder_id':parent['recorder_id'],'source_calendar_rows':len(mapped),'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'train_segment_rows':len(tr),'held_segment_rows':len(he),'max_abs_training_pairwise_Pearson':maxcorr,'candidate_train_unknown_rows':int(feature.loc[train_dates].isna().sum()),'candidate_held_unknown_rows':int(feature.loc[held_dates].isna().sum()),'parent_Recorder_object_loads':4,'source_Recorder_object_loads':1,'DatasetH_builds':1,'model_fits':0,'native_accounts':0,'historical_first_receipt_qualified':False}
(card/'preflight_result.json').write_text(json.dumps(report,indent=2)+'\n')
print({k:report[k] for k in ('status','training_mature_rows','held_mature_rows','max_abs_training_pairwise_Pearson')})

{'status': 'conditional_0946_peer_H5_DatasetH_source_maturity_novelty_preflight_passed_no_fit', 'training_mature_rows': 1055, 'held_mature_rows': 940, 'max_abs_training_pairwise_Pearson': 0.08098169856588952}


In [2]:
import shlex,subprocess,tarfile,tempfile
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings
addendum=json.loads((card/'pre_fit_rehydration_addendum.json').read_text());report=json.loads((card/'preflight_result.json').read_text())
assert addendum['frozen_before_first_fit'] and addendum['original_plan_sha256']==sha(card/'plan.json') and addendum['preflight_receipt_sha256']==sha(card/'preflight_result.json')
assert report['plan_sha256']==sha(card/'plan.json') and report['model_fits']==report['native_accounts']==0
assert report['training_mature_rows']==int(train_dates.sum()) and report['held_mature_rows']==int(held_dates.sum()) and report['max_abs_training_pairwise_Pearson']==maxcorr
assert not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
tr=tr.loc[tr[('meta','train_eligible')].astype(bool)];he=he.loc[he[('meta','held_eligible')].astype(bool)]
assert len(tr)==report['training_mature_rows'] and len(he)==report['held_mature_rows']
ytr=tr[('label','SOXL_relative_H5')].to_numpy(float);yhe=he[('label','SOXL_relative_H5')].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
feature_name=plan['single_mechanism']['feature'];inputs={'baseline':(tr['feature'][base_names].to_numpy(float),he['feature'][base_names].to_numpy(float)),'candidate':(tr['feature'][base_names+[feature_name]].to_numpy(float),he['feature'][base_names+[feature_name]].to_numpy(float))}
assert np.isfinite(inputs['candidate'][0][:,-1]).all() and np.isfinite(inputs['candidate'][1][:,-1]).all()
state={'status':'fit_attempting','plan_sha256':sha(card/'plan.json'),'addendum_sha256':sha(card/'pre_fit_rehydration_addendum.json'),'fit_attempts':[],'fits_completed':[],'bootstrap_attempts':0,'native_accounts':0}
(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
models={};predictions={};tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,rehydration_addendum=addendum,dataset=dataset,derived_coordinate=feature)
 for family in ('baseline','candidate'):
  assert len(state['fit_attempts'])<2 and family not in state['fit_attempts']
  state['fit_attempts'].append(family);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
  model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));model.fit(inputs[family][0],ytr)
  pred=model.predict(inputs[family][1]);assert np.isfinite(pred).all()
  models[family]=model;predictions[family]=pred
  recorder.save_objects(artifact_path='models',**{family:model});recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':pred,'target':yhe})})
  state['fits_completed'].append(family);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');recorder.log_metrics(completed_fits=len(state['fits_completed']))
 dates=pd.DatetimeIndex(he.index.get_level_values('datetime'));scored=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
 scored['paired_mse_improvement']=(scored.target-scored.baseline_prediction)**2-(scored.target-scored.candidate_prediction)**2
 calendar=pd.DataFrame({'signal_date':frame.loc[(frame.index>=pd.Timestamp(cohort['held_reused_diagnostic'][0]))&(frame.index<=pd.Timestamp(cohort['held_reused_diagnostic'][1]))].index})
 full=calendar.merge(scored[['signal_date','paired_mse_improvement']],on='signal_date',how='left',validate='one_to_one')
 assert len(full)==944 and int(full.paired_mse_improvement.notna().sum())==len(scored)==report['held_mature_rows']
 halves=[scored.loc[scored.signal_date.lt(pd.Timestamp('2025-01-01'))],scored.loc[scored.signal_date.ge(pd.Timestamp('2025-01-01'))]]
 annual=[{'year':int(y),'n':int(g.shape[0]),'paired_mean_mse_improvement':float(g.paired_mse_improvement.mean())} for y,g in scored.groupby(scored.signal_date.dt.year,sort=True)]
 complete=[x for x in annual if x['year'] in cohort['complete_held_years']]
 assert len(complete)==2 and all(x['n']>=240 for x in complete)
 boot=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempts']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 interval,indices,replicates=full_calendar_bootstrap(full.paired_mse_improvement.to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws'],seed=boot['seed'],alpha=boot['one_sided_alpha'])
 assert interval['valid_replicates']>=9990
 coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
 metrics={'baseline_held_mse':float(np.mean((scored.target-scored.baseline_prediction)**2)),'candidate_held_mse':float(np.mean((scored.target-scored.candidate_prediction)**2)),'paired_mean_mse_improvement':float(scored.paired_mse_improvement.mean()),'held_half_means':[float(x.paired_mse_improvement.mean()) for x in halves],'annual':annual,'added_standardized_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
 gates={'coefficient_positive':coefficient>0,'full_held_mean_positive':metrics['paired_mean_mse_improvement']>0,'both_held_halves_mean_positive':all(x>0 for x in metrics['held_half_means']),'each_complete_2024_2025_year_mean_positive':all(x['paired_mean_mse_improvement']>0 for x in complete),'bootstrap_lower_positive':metrics['bootstrap_lower']>0};passed=bool(all(gates.values()))
 recorder.save_objects(artifact_path='analysis',scored=scored,full_eval_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=replicates,bootstrap_indices=indices)
 recorder.log_metrics(paired_mse_improvement=metrics['paired_mean_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'decision':'conditional_development_0946_peer_breadth_SOXL_relative_H5_information_pass_first_receipt_pending_no_account' if passed else 'not_promoted_fixed_0946_peer_breadth_SOXL_relative_H5_information_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'preflight':report,'information_gate_pass':passed,'metrics':metrics,'gates':gates,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'held_market_history_reused':True,'historical_first_receipt_qualified':False,'independent_forward_qualified_days':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'artifact_policy':plan['artifact_policy'],'limitations':['Original 2018-2022 training and reused 2023-2026 held data do not create independent forward evidence.','The 09:45 bar clock is assumed; actual historical first receipt and finality are unqualified.','Information MSE is not a funded 0bp Tiered Stage1 account or SOXX net result.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='opening-peer-info-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:transfer=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert transfer.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  members=[m for m in source.getmembers() if m.isfile()];names=[m.name for m in members]
  assert any(x.endswith('/artifacts/models/baseline') for x in names) and any(x.endswith('/artifacts/models/candidate') for x in names)
  for member in members:
   local=card/member.name;assert local.is_file() and source.extractfile(member).read()==local.read_bytes()
summary['nas_evidence']=evidence;summary['nas_scoped_restore_verified']=True;(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(summary['decision'],metrics,gates,state['recorder_id'])

not_promoted_fixed_0946_peer_breadth_SOXL_relative_H5_information_failed {'baseline_held_mse': 0.01150004118040791, 'candidate_held_mse': 0.011502326536765957, 'paired_mean_mse_improvement': -2.2853563580463027e-06, 'held_half_means': [-1.1083991420152717e-06, -3.6342890576983076e-06], 'annual': [{'year': 2023, 'n': 250, 'paired_mean_mse_improvement': 3.6722882828149446e-08}, {'year': 2024, 'n': 252, 'paired_mean_mse_improvement': -2.244432896820251e-06}, {'year': 2025, 'n': 250, 'paired_mean_mse_improvement': -1.7840735573229227e-06}, {'year': 2026, 'n': 188, 'paired_mean_mse_improvement': -6.094682010325149e-06}], 'added_standardized_coefficient': 0.0008665540062239291, 'bootstrap_lower': -1.673079801700551e-05, 'bootstrap_upper': 1.2279053732243563e-05, 'bootstrap': {'paired_mean_loss_improvement': -2.285356358046304e-06, 'lower_bound': -1.673079801700551e-05, 'upper_bound': 1.2279053732243563e-05, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 944, 'scored_row

## 中文结论
# 原 Stage1 09:46 同业早盘广度：SOXL 相对收益信息检验

冻结原 39 项 Stage1 输入为对照，只增加 AMD/NVDA/SMH 前 15 分钟同涨比例一个坐标。2018–2022 成熟训练日 1055、2023–2026 复用历史成熟评价日 940；训练期与旧输入最大逐项相关绝对值 0.0810。先核对 Qlib 来源、标签成熟与同日 09:45/09:46 时钟，再固定两次 Ridge 拟合及一次全日历区块检验。

评价 MSE：旧 39 项 0.011500041，新增坐标 0.011502327，成对改善 -2.28535636e-06。新增标准化系数 +0.000866554006 符合预定正号，但两个半段改善分别为 -1.10839914e-06、-3.63428906e-06；2024/2025 完整年也均为负，校正区块下界 -1.6730798e-05。仅符号门通过，其余全部失败，决定 `not_promoted_fixed_0946_peer_breadth_SOXL_relative_H5_information_failed`。不改用相反符号、其他时钟或年份救援，0 新账户。

来源 bar 完成时间是假设，历史首次收取和最终版本未合格；复用历史的信息损失也不等于费用后完整 Stage1 账户或独立前向证据。Recorder `26fe25ee046447c3b08c8b4ab142deb3` 定向读回 944 个评价日、940 个成熟标签，NAS SHA-256 `1a7e263dee2070c195f64aa40f3983b65013274ebca027c2c47e2c02afbf43a6` 经远端哈希和限定逐文件恢复。SOXX v2 目标仍未达成，下一机制须带来不同的决策前信息；原 Stage1 父方案、0bp 首检与 2/5bp/前向门保持不变。
